# Voice Engine

An educational, step-by-step **multi-voice engine** built with Python, NumPy and PyTorch.
This single notebook walks through the whole pipeline (Phases 1–9), importing all logic from `src/`.

> **This project does NOT synthesize speech.** It ends at a *conditioning representation* that a future voice model could use.
> Sections 5-11 use the **untrained** Phase 3 encoder (architecture demo); section 12 **trains** a real (experimental) encoder.

Works on **CPU** (Windows or Google Colab). A GPU is optional.

## 1. Project Overview

```
Audio ─► Speaker Encoder ─► Speaker Embedding ─► Voice Registry ─► Voice Engine ─► Voice Conditioning ─► (future voice model)
```

| Section | Phase | Topic |
|---|---|---|
| 2 | 1 | Audio inspection |
| 3–4 | 2 | Waveform, spectrogram, Mel spectrogram, audio features |
| 5 | 3 | Speaker embeddings (untrained encoder) |
| 6 | 4 | Speaker verification (demonstration) |
| 7 | 5 | Voice Registry |
| 8 | 6 | Multi-Voice Engine |
| 9 | 7 | Voice Conditioning |
| 10 | 8 | Production architecture |
| 11 | — | End-to-end demo |
| 12 | 9 | Real (trainable) speaker embedding model |

### Setup
Run this first. In **Colab** it clones the repository and installs the requirements;
locally it just makes sure we are in the project root.

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("Voice_engine").exists():
        subprocess.run(["git", "clone", "https://github.com/8919134556/Voice_engine.git"], check=True)
    os.chdir("Voice_engine")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")                      # run from the project root
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Project root:", Path.cwd())

In [ ]:
import tempfile
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.config import settings
from src.models.speaker_encoder import get_device
from src.utils.logging import setup_logging

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = get_device()                   # GPU if available, otherwise CPU (both work)
print("Device:", DEVICE)
print(f"Settings: sample rate {settings.SAMPLE_RATE} Hz, embedding dimension {settings.EMBEDDING_DIMENSION}, "
      f"default language '{settings.DEFAULT_LANGUAGE}'")

### Your data (private) or synthetic data

By default the notebook looks for `dataset/speaker_01/*.wav` in the project. **Your recordings are never in GitHub**, so:

- **Locally:** put your WAV files in `dataset/speaker_01/` (git-ignored).
- **Colab:** upload them to your private Google Drive and set `DATASET_DIR` to that folder, e.g.
  `from google.colab import drive; drive.mount("/content/drive")` then
  `DATASET_DIR = Path("/content/drive/MyDrive/voice_engine/dataset")`.

If no WAV files are found, the notebook creates **synthetic speech-like audio** for two fake speakers, so everything still runs.
All files this notebook writes go to a temporary folder (`WORK`), never into your project.

In [ ]:
from src.utils.synthetic import write_synthetic_dataset

DATASET_DIR = settings.DATASET_DIR      # change this to your private folder if needed
WORK = Path(tempfile.mkdtemp(prefix="voice_engine_notebook_"))
USING_SYNTHETIC_AUDIO = False

if not list((Path(DATASET_DIR) / "speaker_01").glob("*.wav")):
    DATASET_DIR = write_synthetic_dataset(WORK / "dataset", {"speaker_01": 120.0, "speaker_02": 210.0})
    USING_SYNTHETIC_AUDIO = True
    print("No recordings found -> using SYNTHETIC audio (two fake speakers).")
DATASET_DIR = Path(DATASET_DIR)
SPEAKERS = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
print("Dataset:", DATASET_DIR)
print("Speakers:", SPEAKERS)
print("Working folder (temporary):", WORK)

## 2. Audio Inspection  *(Phase 1)*

```
WAV file ─► load_audio() ─► tensor [channels, samples] ─► inspect_audio() ─► report + quality checks
```
Checks: mono/stereo, 16 kHz or not, empty/corrupted, silent, unusually short, clipping.

In [ ]:
from src.audio.loader import list_audio_files, load_audio, load_mono_16k
from src.audio.inspect import inspect_audio, format_report

wav_files = list_audio_files(DATASET_DIR / "speaker_01")
for path in wav_files:
    print(format_report(inspect_audio(path)), end="\n\n")

In [ ]:
# Every file is converted to the project standard: mono, 16 kHz
waveform, sr = load_audio(wav_files[0])
mono16k, sr16k = load_mono_16k(wav_files[0])
print("raw:          ", tuple(waveform.shape), sr, "Hz")
print("mono + 16 kHz:", tuple(mono16k.shape), sr16k, "Hz")

## 3. Waveform and Spectrogram  *(Phase 2)*

- **Waveform:** amplitude over time. Flat = silence, tall = loud, dense small wiggles = hissy sounds like "s".
- **STFT spectrogram:** cut into 25 ms windows every 10 ms and measure the frequencies in each → time × frequency × energy.
- **Mel spectrogram:** 80 triangular filters squeeze the frequency axis the way human hearing does
  (fine detail at low Hz, coarse at high Hz). This is the standard input for speech neural networks.

In [ ]:
from src.audio import features as F
from src.visualization import plots

waveform, sr = load_mono_16k(wav_files[0])
plots.plot_waveform(waveform, sr, f"Waveform — {wav_files[0].name}");

In [ ]:
power = F.stft_power(waveform)
print("Spectrogram:", tuple(power.shape), "= (frequency bins, time frames)")
plots.plot_spectrogram(F.power_to_db(power), sr, title="STFT spectrogram");

In [ ]:
plots.plot_mel_filterbank(F.mel_filterbank(sr), sr);
mel = F.mel_spectrogram(waveform, sr)
print("Mel spectrogram:", tuple(mel.shape), "= (Mel bands, time frames)")
plots.plot_mel_spectrogram(F.power_to_db(mel), sr, title="Mel spectrogram");

## 4. Audio Features  *(Phase 2)*

| Feature | Meaning |
|---|---|
| duration | seconds |
| RMS | overall loudness |
| ZCR | how often the signal crosses zero (high for hiss/noise) |
| spectral centroid | "centre of mass" of the spectrum in Hz (brightness) |

These describe the **recording**, not the **speaker**: they change with volume, microphone and words.

In [ ]:
print(f"{'file':<16}{'duration':>9}{'RMS':>9}{'ZCR':>9}{'centroid Hz':>13}")
for path in wav_files:
    w, s = load_mono_16k(path)
    f = F.extract_features(w, s)
    print(f"{path.name:<16}{f['duration']:>9.2f}{f['rms']:>9.4f}{f['zcr']:>9.4f}{f['spectral_centroid']:>13.1f}")

print("\nSame audio, quieter / with hiss:")
for name, w in [("original", waveform), ("30% volume", waveform * 0.3),
                ("+ hiss", waveform + 0.02 * torch.randn_like(waveform))]:
    f = F.extract_features(w, sr)
    print(f"  {name:<11} RMS={f['rms']:.4f}  ZCR={f['zcr']:.4f}  centroid={f['spectral_centroid']:.0f} Hz")

## 5. Speaker Embedding  *(Phase 3)*

```
Mel [1, 1, 80, time] ─► Conv2D ─► ReLU ─► Conv2D ─► ReLU ─► AdaptivePool ─► Flatten ─► Linear ─► 128 numbers ─► L2-normalize
```
A **speaker embedding** is a fixed-length vector meant to describe *who* is speaking.

> ⚠️ The encoder has **random weights**. It produces 128 numbers for any input, but they are **not** a meaningful
> speaker embedding until the network is trained on many speakers.

In [ ]:
from src.models.speaker_encoder import SpeakerEncoder, count_parameters
from src.embeddings.generate_embeddings import wav_to_encoder_input, generate_for_speaker, embed_mel, waveform_to_encoder_input

torch.manual_seed(settings.ENCODER_SEED)
encoder = SpeakerEncoder().to(DEVICE).eval()
print(f"SpeakerEncoder: {count_parameters(encoder):,} parameters (UNTRAINED)")

x = wav_to_encoder_input(wav_files[0]).to(DEVICE)
print("input          ", list(x.shape))
with torch.no_grad():
    x = encoder.relu(encoder.conv1(x)); print("conv1 + ReLU   ", list(x.shape))
    x = encoder.relu(encoder.conv2(x)); print("conv2 + ReLU   ", list(x.shape))
    x = encoder.pool(x);                print("adaptive pool  ", list(x.shape), "<- time averaged away")
    x = encoder.fc(encoder.flatten(x)); print("linear         ", list(x.shape), "<- 128-D embedding")

In [ ]:
# Embed every recording of every speaker -> WORK/embeddings/<speaker>/<file>.npy (temporary, private)
recording_embeddings = {}
for speaker in SPEAKERS:
    for name, emb in generate_for_speaker(encoder, DATASET_DIR / speaker, WORK / "embeddings", DEVICE, verbose=False).items():
        recording_embeddings[f"{speaker}/{Path(name).stem}"] = emb
for key, emb in recording_embeddings.items():
    print(f"{key}: shape {emb.shape}, L2 norm {np.linalg.norm(emb):.3f}")

In [ ]:
from src.similarity.cosine import cosine_similarity

# Proof that the untrained vectors are NOT speaker embeddings: non-voice sounds look "similar" too.
t = torch.arange(3 * settings.SAMPLE_RATE) / settings.SAMPLE_RATE
first_key = next(iter(recording_embeddings))
for name, w in [("white noise", 0.1 * torch.randn(1, len(t))),
                ("440 Hz beep", 0.3 * torch.sin(2 * torch.pi * 440 * t).unsqueeze(0))]:
    e = embed_mel(encoder, waveform_to_encoder_input(w, settings.SAMPLE_RATE), DEVICE)
    print(f"{first_key} vs {name}: {cosine_similarity(recording_embeddings[first_key], e):.4f}")

plots.plot_embeddings_pca(recording_embeddings, title="Recording embeddings (untrained, PCA to 2-D — for viewing only)");

## 6. Speaker Verification  *(Phase 4)*

**Verification** asks *"is this the claimed speaker?"* (yes/no), unlike **identification** (*"who is this?"*).

```
Audio A ─► encoder ─► emb A ─┐
                             ├─► cosine similarity ─► ≥ threshold ? ─► MATCH / NOT MATCH
Audio B ─► encoder ─► emb B ─┘
```
- **Genuine pair:** same speaker. **Impostor pair:** different speakers.
- **False accept:** impostor accepted. **False reject:** real speaker rejected.
- The threshold (`VERIFICATION_THRESHOLD = 0.70`) is an **educational example**, not calibrated.

> DEMONSTRATION ONLY — the encoder is not trained.

In [ ]:
from itertools import combinations
from src.verification.verifier import create_verifier, UNTRAINED_WARNING

verifier = create_verifier(device=DEVICE)       # same seed -> same weights as the encoder above
print(UNTRAINED_WARNING, "\n")
rows = []
for a, b in combinations(recording_embeddings, 2):
    kind = "genuine" if a.split("/")[0] == b.split("/")[0] else "impostor"
    result = verifier.verify(recording_embeddings[a], recording_embeddings[b], settings.VERIFICATION_THRESHOLD)
    rows.append((a, b, kind, result.similarity))
    print(f"{a} vs {b}  {kind:<8}  {result.similarity:.4f}  {result.decision}")

impostor = [s for *_, k, s in rows if k == "impostor"]
if impostor:
    far = np.mean([s >= settings.VERIFICATION_THRESHOLD for s in impostor])
    print(f"\nFalse Acceptance Rate at {settings.VERIFICATION_THRESHOLD}: {far:.0%}  <- the untrained encoder accepts impostors")
else:
    print("\nNo impostor pairs (only one speaker) -> FAR cannot be measured.")
plots.plot_pair_scores(rows, settings.VERIFICATION_THRESHOLD, title="Pair scores (UNTRAINED — demo only)");

**How a real threshold is chosen** — the next cell uses **made-up (hypothetical) scores**, NOT your data,
to show what a *trained* system's genuine/impostor distributions look like and how the threshold trades
FAR against FRR. The **EER** is where FAR = FRR.

In [ ]:
rng = np.random.default_rng(0)
genuine_scores = rng.normal(0.75, 0.08, 500)     # HYPOTHETICAL
impostor_scores = rng.normal(0.35, 0.12, 500)    # HYPOTHETICAL
thresholds = np.linspace(0, 1, 501)
far = np.array([(impostor_scores >= t).mean() for t in thresholds])
frr = np.array([(genuine_scores < t).mean() for t in thresholds])
i = np.argmin(np.abs(far - frr))
print(f"EER ~ {(far[i] + frr[i]) / 2:.1%} at threshold {thresholds[i]:.2f}  (hypothetical data)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.hist(genuine_scores, bins=40, alpha=0.6, label="genuine"); ax1.hist(impostor_scores, bins=40, alpha=0.6, label="impostor")
ax1.axvline(thresholds[i], color="red", ls="--"); ax1.set_title("HYPOTHETICAL scores"); ax1.legend()
ax2.plot(thresholds, far, label="FAR"); ax2.plot(thresholds, frr, label="FRR"); ax2.set_xlabel("threshold"); ax2.legend()
plt.tight_layout()

## 7. Voice Registry  *(Phase 5)*

```
VOICE REGISTRY (voice_profiles.json — metadata only)
   ├── voice_001 ─► metadata ─► embeddings/voice_001.npy
   └── voice_002 ─► metadata ─► embeddings/voice_002.npy
```
- **`voice_id`** is the identity (unique, permanent). **Gender is optional metadata only.**
- Embeddings stay in separate `.npy` files; the JSON only stores their paths.
- `register` rejects duplicates and invalid embeddings; `update` changes only the given fields; `remove` keeps the file.

In [ ]:
from src.voice_registry import (VoiceProfile, VoiceRegistry, average_embeddings,
                                DuplicateVoiceError, InvalidEmbeddingError)
from src.utils.synthetic import random_embedding

registry = VoiceRegistry(WORK / "voice_profiles.json")

# voice_001 = average of speaker_01's recording embeddings (the "speaker profile" idea)
average_embeddings(sorted((WORK / "embeddings" / "speaker_01").glob("*.npy")), WORK / "embeddings" / "voice_001.npy")
registry.register(VoiceProfile(voice_id="voice_001", name="Voice A", gender="male", languages=["en", "hi"],
                               description="from speaker_01", embedding_path="embeddings/voice_001.npy"))

# voice_002 = speaker_02 if we have one, otherwise a clearly FAKE random vector
if "speaker_02" in SPEAKERS:
    average_embeddings(sorted((WORK / "embeddings" / "speaker_02").glob("*.npy")), WORK / "embeddings" / "voice_002.npy")
else:
    np.save(WORK / "embeddings" / "voice_002.npy", random_embedding(seed=2))
registry.register(VoiceProfile(voice_id="voice_002", name="Voice B", gender="female", languages=["en"],
                               embedding_path="embeddings/voice_002.npy"))

for bad in [VoiceProfile(voice_id="voice_001", name="Dup", embedding_path="embeddings/voice_002.npy"),
            VoiceProfile(voice_id="voice_003", name="Missing", embedding_path="embeddings/nope.npy")]:
    try:
        registry.register(bad)
    except (DuplicateVoiceError, InvalidEmbeddingError) as exc:
        print("REJECTED:", exc)

registry.update("voice_001", name="Voice A (renamed)")
registry.save()
reloaded = VoiceRegistry(WORK / "voice_profiles.json")
print("\nAfter save + reload:", [(p.voice_id, p.name, p.languages) for p in reloaded.list_voices()])
print("Hindi voices:", [p.voice_id for p in reloaded.find_by_language("hi")])
print("\nvoice_profiles.json:\n" + (WORK / "voice_profiles.json").read_text()[:600], "...")

## 8. Multi-Voice Engine  *(Phase 6)*

**One** `VoiceEngine` manages **many** voices — no `MaleVoiceEngine` or `HindiVoiceEngine`.
It uses the registry (no second database), remembers the selected voice, and caches embeddings:

```
first request:  voice_001 ─► disk ─► memory cache        later requests: voice_001 ─► memory cache
```

In [ ]:
from src.engine import VoiceEngine, VoiceRequest, VoiceNotFoundError, VoiceLanguageNotSupportedError

setup_logging("INFO")      # show the engine's log messages from now on
engine = VoiceEngine(reloaded)
print("Voices:", [p.voice_id for p in engine.list_voices()])

engine.select_voice("voice_001")
engine.get_embedding("voice_001")       # cache miss -> disk
engine.get_embedding("voice_001")       # cache hit  -> memory
same_engine = id(engine)
engine.select_voice("voice_002")        # switching = only the selected id changes
print("Current:", engine.current_voice_id, "| same engine object:", id(engine) == same_engine)

selected = engine.resolve(VoiceRequest(voice_id="voice_001", language="hi"))
print("Resolved:", selected.voice_id, selected.profile.name, selected.embedding.shape, selected.language)

for request in [VoiceRequest("voice_002", "hi"), VoiceRequest("voice_999")]:
    try:
        engine.resolve(request)
    except (VoiceLanguageNotSupportedError, VoiceNotFoundError) as exc:
        print("REJECTED:", exc)
print("Selection unchanged after failures:", engine.current_voice_id)

## 9. Voice Conditioning  *(Phase 7)*

```
VoiceProfile      = identity + metadata
Speaker embedding = numbers on disk
VoiceConditioning = validated, L2-normalized, in-memory COPY, ready for a future model
```
L2 normalization: `v / ||v||` → same direction, length 1. The stored `.npy` and the engine cache are never modified.

In [ ]:
c1 = engine.condition("voice_001")                               # facade -> VoiceConditioner
c2 = engine.condition(VoiceRequest("voice_002", "en"))
for c in (c1, c2):
    print(f"Voice ID: {c.voice_id}  dimension: {c.dimension}  normalized: {c.normalized}  L2 norm: {c.l2_norm:.4f}")
stored = np.load(WORK / "embeddings" / "voice_001.npy")
print("stored file still identical to the engine cache:", np.array_equal(stored, engine.get_embedding("voice_001")))

## 10. Production Architecture  *(Phase 8)*

```
                    Application
                         │
                         ▼
               VoiceEngine  (facade)
                         │
              ┌──────────┴──────────┐
              ▼                     ▼
   VoiceProvider (interface)   VoiceConditioner
     = VoiceRegistry today           │
              │                      ▼
              ▼               Speaker Embedding
        VoiceProfile                 │
              └──────────┬───────────┘
                         ▼
                (Future Voice Model)  ─►  (Future Audio)
```

| Component | Responsibility |
|---|---|
| `VoiceRegistry` | storage + metadata (JSON + `.npy`) |
| `VoiceEngine` | orchestration: lookup, selection, cache, requests |
| `VoiceConditioner` | prepare the speaker representation |
| `SpeakerEncoder` | generate embeddings from audio |
| `verification` | compare speakers |
| `config.settings` | all important constants in one place |
| `utils.logging` | standard logging (voice ids only — never raw embeddings or audio) |

**`VoiceProvider`** is a small Protocol: any store with `list_voices / exists / get / load_embedding / embedding_dim`
works — JSON today, a database or an API later — without changing the engine.

In [ ]:
from src.engine import VoiceProvider
print("VoiceRegistry is a VoiceProvider:", isinstance(reloaded, VoiceProvider))
print("info():        ", engine.info())
print("health_check():", engine.health_check())

## 11. End-to-End Demo

```
Audio ─► Speaker Encoder ─► Embedding ─► Voice Profile ─► Voice Registry ─► Voice Engine ─► Voice Selection ─► Voice Conditioning
```
One cell, a fresh folder, using only the public APIs. The result is a **conditioning representation — no speech is generated.**

In [ ]:
e2e = Path(tempfile.mkdtemp(prefix="voice_engine_e2e_"))

# Audio (synthetic, never personal)
dataset = write_synthetic_dataset(e2e / "dataset", {"speaker_01": 130.0}, files_per_speaker=2)
# Speaker Encoder -> Embedding
torch.manual_seed(settings.ENCODER_SEED)
generate_for_speaker(SpeakerEncoder().to(DEVICE), dataset / "speaker_01", e2e / "embeddings", DEVICE, verbose=False)
average_embeddings(sorted((e2e / "embeddings" / "speaker_01").glob("*.npy")), e2e / "embeddings" / "voice_001.npy")
# Voice Profile -> Voice Registry
e2e_registry = VoiceRegistry(e2e / "voice_profiles.json")
e2e_registry.register(VoiceProfile(voice_id="voice_001", name="E2E Voice", languages=["en"],
                                   embedding_path="embeddings/voice_001.npy"))
e2e_registry.save()
# Voice Engine -> Voice Selection -> Voice Conditioning
e2e_engine = VoiceEngine.from_registry_file(e2e / "voice_profiles.json", default_voice_id="voice_001")
conditioning = e2e_engine.condition(VoiceRequest("voice_001", settings.DEFAULT_LANGUAGE))

print("\nResult:", f"VoiceConditioning(voice_id={conditioning.voice_id!r}, dimension={conditioning.dimension}, "
      f"normalized={conditioning.normalized}, language={conditioning.language!r})")
print("Health:", e2e_engine.health_check()["status"])

## 12. Real Speaker Embedding Model  *(Phase 9)*

The Phase 3 encoder had **random weights**: it turned any sound into 128 numbers, but those numbers did not
describe *who* was speaking (noise scored ~0.97 against a voice). Phase 9 **trains** an encoder.

**Why several speakers?** The model learns by telling speakers apart. With one speaker there is nothing to
contrast, so training needs **at least 2 speakers** (realistically 10+).

```
                    ┌──► classification head ─► speaker id ─► CrossEntropy loss   (training only)
Mel features ─► Conv-BN-ReLU ─► Conv-BN-ReLU ─► Conv-ReLU ─► pool ─► Linear ─► 128-D ─┤
                    └──► L2-normalize ─► speaker embedding                            (kept after training)
```

To classify the training speakers, the encoder must pack speaker-specific information into its 128 numbers.
After training, the head is discarded and the embedding is used for verification, the registry and conditioning.

> This is an **experimental** speaker embedding model and **not** a production-grade speaker recognition system.

In [ ]:
from src.training import (discover_recordings, split_recordings, TrainingConfig, TrainableSpeakerEncoder,
                          count_parameters, train_speaker_encoder, get_device, describe_device)
from src.utils.synthetic import write_synthetic_speaker_dataset

# 1-3. Dataset inspection. Real data needs >= 2 speaker folders; otherwise use FAKE speakers to learn the pipeline.
recordings, speakers = discover_recordings(DATASET_DIR)
# (the tiny 3-file synthetic set from the setup cell is too small to train on, so it is replaced as well)
USING_SYNTHETIC_SPEAKERS = USING_SYNTHETIC_AUDIO or len(speakers) < settings.MIN_SPEAKERS_FOR_TRAINING
if USING_SYNTHETIC_SPEAKERS:
    TRAIN_DATASET = write_synthetic_speaker_dataset(WORK / "speaker_dataset", n_speakers=6, files_per_speaker=12)
    recordings, speakers = discover_recordings(TRAIN_DATASET)
    print("*** Using a SYNTHETIC multi-speaker dataset (fake voices) - results only show that the pipeline works. ***\n")
else:
    TRAIN_DATASET = DATASET_DIR
print(f"Speakers:   {len(speakers)}")
print(f"Recordings: {len(recordings)}")
for name, label in speakers.items():
    print(f"  {name} -> class {label}: {sum(r.speaker == name for r in recordings)} files")

In [ ]:
# 4-5. One example recording: waveform and Mel spectrogram (the model's input representation)
example = recordings[0]
w, s = load_mono_16k(example.path)
plots.plot_waveform(w, s, f"Waveform - {example.speaker}/{example.path.name}");
plots.plot_mel_spectrogram(F.power_to_db(F.mel_spectrogram(w, s)), s, title="Mel spectrogram (model input, before normalization)");

In [ ]:
# 6-7. Model + training configuration (defaults from src/config/settings.py; fewer epochs here to keep it quick)
DEVICE = get_device()
print(describe_device(DEVICE))
splits, split_warnings = split_recordings(recordings)
print(f"Split (per speaker, by recording - no file in two splits): "
      f"train {len(splits['train'])}, val {len(splits['val'])}, test {len(splits['test'])}")
for warning in split_warnings:
    print("  [WARN]", warning)
config = TrainingConfig(epochs=20)
print(f"Model: TrainableSpeakerEncoder with {count_parameters(TrainableSpeakerEncoder()):,} parameters -> "
      f"{config.embedding_dim}-D embedding")
print("Config:", config.to_dict())

In [ ]:
# 8. Training (best model by validation loss -> WORK/checkpoints/best_model.pt; use the script to keep it in outputs/)
result = train_speaker_encoder(splits, speakers, config, DEVICE, checkpoint_dir=WORK / "checkpoints")
print("Best epoch:", result.best_epoch)

In [ ]:
# 9. Training curves
plots.plot_training_history(result.history);

In [ ]:
# 10-11. Evaluation on the TEST split: classification accuracy + genuine vs impostor cosine similarity
from src.training.evaluate import evaluate_encoder

evaluation = evaluate_encoder(result.classifier, splits["test"], config.segment_frames, DEVICE)
report, arrays = evaluation["report"], evaluation["arrays"]
print(f"Test classification accuracy: {report['test_accuracy']:.3f}")
for kind in ("genuine", "impostor"):
    r = report[kind]
    print(f"{kind:<9} pairs: {r['count']:>4}   mean similarity: " + (f"{r['mean']:.3f}" if r["count"] else "n/a"))
print("\nthreshold   FAR    FRR    accuracy")
for row in report["thresholds"][::2]:
    print(f"   {row['threshold']:.2f}    {row['far']:.3f}  {row['frr']:.3f}  {row['accuracy']:.3f}")
if report["eer"] is not None:
    print(f"\nEER {report['eer']:.3f} at threshold {report['eer_threshold']:.3f}  "
          f"(from only {report['genuine']['count']} genuine / {report['impostor']['count']} impostor pairs - a rough indication)")
plots.plot_similarity_distributions(arrays["genuine"], arrays["impostor"], report["thresholds"], report["eer_threshold"]);

In [ ]:
# 12. PCA of the test embeddings, coloured by speaker (qualitative only)
names = {label: name for name, label in speakers.items()}
plots.plot_speaker_embeddings_pca(arrays["embeddings"], arrays["labels"], names,
                                  title="Trained test embeddings (PCA, qualitative only)");

In [ ]:
# 13. Example verification with the TRAINED encoder (reuses the Phase 4 verifier)
trained_verifier = create_verifier(checkpoint=result.best_checkpoint, device=DEVICE)
by_speaker = {}
for r in splits["test"]:
    by_speaker.setdefault(r.speaker, []).append(r)
spk = list(by_speaker)
threshold = report["eer_threshold"] or settings.VERIFICATION_THRESHOLD
pairs = []
if len(by_speaker[spk[0]]) >= 2:
    pairs.append(("same speaker", by_speaker[spk[0]][0], by_speaker[spk[0]][1]))
pairs.append(("different speakers", by_speaker[spk[0]][0], by_speaker[spk[1]][0]))
for kind, a, b in pairs:
    v = trained_verifier.verify_files(a.path, b.path, threshold)
    print(f"{kind:<19} {a.speaker}/{a.path.name} vs {b.speaker}/{b.path.name}: "
          f"similarity {v.similarity:.3f} -> {v.decision} (threshold {threshold:.3f})")

### 14. Limitations (read these)

- **Experimental model, not production-grade.** It is small (~160k parameters) and trained on very little data.
- **Synthetic data** (used when fewer than 2 real speakers exist) only proves the pipeline runs — it says nothing about real voices.
- **Test protocol:** val/test are *new recordings of training speakers*. Real systems are evaluated on *unseen* speakers.
- **Few test pairs** make FAR/FRR/EER very noisy; the EER threshold here is **not** a universal threshold.
- **Better encoders** (future): angular-margin losses (AAM-Softmax/ArcFace), triplet/contrastive training, and
  architectures such as ECAPA-TDNN, x-vectors or ResNet speaker encoders trained on thousands of speakers.

### Clean up
Removes the temporary folders created by this notebook (they contain only synthetic or derived data).

In [ ]:
import shutil
setup_logging("WARNING")
for folder in (WORK, e2e):
    shutil.rmtree(folder, ignore_errors=True)
print("Temporary folders removed.")